# Predicting Food Insecurity Risk from Conflict Dynamics in South Sudan

**Author:** [Your full name]  
**Module:** Introduction to Machine Learning — Summative Project  
**Date:** [Submission date]

**GitHub:** [your repo link]  
**Demo video:** [your video link]  
**Written report:** [your report link]

---

**Mission alignment:** This project supports the goal of fostering enduring peace and sustainable development in South Sudan and Africa at large. By investigating whether county-level conflict patterns can anticipate food insecurity crises, it contributes to earlier, more targeted humanitarian and peacebuilding response.

**Problem statement:** South Sudan has experienced near-continuous armed conflict since independence in 2011, and routinely faces acute food insecurity affecting millions. This project asks: *can conflict event data predict the likelihood of a food security crisis at the county level one month in advance?* A reliable early-warning signal would allow humanitarian actors to pre-position resources and prioritise peacebuilding interventions before crises escalate.

## 0. Setup and Reproducibility

All dependencies are installed below. Random seeds are fixed before any data is loaded so results are fully reproducible from a fresh kernel.

In [ ]:
# !pip install pandas numpy scikit-learn tensorflow matplotlib seaborn --quiet

import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

import sys
sys.path.insert(0, '..')
from src.data_loading import load_acled, load_ipc
from src.preprocessing import (
    standardize_county_names, filter_ipc, find_unmatched_counties,
    aggregate_acled_to_county_month, pivot_ipc_wide, merge_conflict_food_security,
)

print('TensorFlow version:', tf.__version__)
print('Setup complete. SEED =', SEED)

## 1. Data Loading

Two datasets are used:
- **ACLED**: event-level records of all political violence and protest activity in South Sudan from May 2011 to June 2025. Each row is one conflict event with location, actor, event type, and fatality count.
- **FEWS NET IPC**: monthly food security phase assessments for South Sudan counties from January 2017 to April 2026, measuring what percentage of each county's population falls into IPC Phases 1-5 (Phase 3+ = Crisis or worse).

Full attribution and download instructions: see `data/README.md`.

In [ ]:
acled_raw = load_acled('../data/raw/acled_south_sudan.csv')
ipc_raw   = load_ipc('../data/raw/fewsnet_ipc_south_sudan.csv')

print('ACLED shape:', acled_raw.shape)
print('ACLED date range:', acled_raw['event_date'].min().date(), '->', acled_raw['event_date'].max().date())
print()
print('IPC shape:', ipc_raw.shape)
print('IPC date range:', ipc_raw['From'].min().date(), '->', ipc_raw['From'].max().date())
print()
print('ACLED missing values (non-zero only):')
miss = acled_raw.isnull().sum()
print(miss[miss > 0])
print('IPC missing values:', ipc_raw.isnull().sum().sum())

**[TODO - interpret after running.]** Note the date gap in ACLED (ends 2025, not 2026), what the structural missingness in assoc_actor_1/actor2 means, and why zero missing values in IPC is expected.

## 2. Exploratory Data Analysis

In [ ]:
# 2a. Events over time
fig, ax = plt.subplots(figsize=(13,4))
acled_raw.set_index('event_date').resample('M').size().plot(ax=ax, color='steelblue', linewidth=1.2)
ax.set_title('Monthly Conflict Event Count - South Sudan (ACLED)', fontsize=13)
ax.set_xlabel('Date'); ax.set_ylabel('Events per month')
plt.tight_layout()
plt.savefig('../reports/figures/01_events_per_month.png', dpi=150)
plt.show()

**[TODO - interpret.]** When do event counts peak? Do spikes correspond to the 2013 civil war outbreak, 2016 Juba clashes, or peace agreement periods?

In [ ]:
# 2b. Event type distribution
fig, ax = plt.subplots(figsize=(8,4))
acled_raw['event_type'].value_counts().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('ACLED Event Type Distribution - South Sudan', fontsize=13)
ax.set_xlabel('Count')
plt.tight_layout()
plt.savefig('../reports/figures/02_event_type_distribution.png', dpi=150)
plt.show()

**[TODO - interpret.]** Which event type dominates? What does a high proportion of Violence against civilians vs Battles suggest about the conflict's nature?

In [ ]:
# 2c. Fatalities distribution
fig, axes = plt.subplots(1,2, figsize=(12,4))
acled_raw['fatalities'].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Fatalities per Event (raw)'); axes[0].set_xlabel('Fatalities')
np.log1p(acled_raw['fatalities']).hist(bins=50, ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Fatalities per Event (log1p)')
plt.tight_layout()
plt.savefig('../reports/figures/03_fatalities_distribution.png', dpi=150)
plt.show()
print('Max:', acled_raw['fatalities'].max(), ' Mean:', round(acled_raw['fatalities'].mean(),2),
      ' Zero-fatality events:', (acled_raw['fatalities']==0).sum())

**[TODO - interpret.]** Is the distribution right-skewed? Does this motivate log-transforming fatalities features?

In [ ]:
# 2d. IPC Phase 3+ trend
ipc_cur_eda = ipc_raw[ipc_raw['Validity period']=='current'].copy()
ipc_3plus   = ipc_cur_eda[ipc_cur_eda['Phase']=='3+'][['From','Percentage']].groupby('From')['Percentage'].mean().reset_index()

fig, ax = plt.subplots(figsize=(13,4))
ax.plot(ipc_3plus['From'], ipc_3plus['Percentage'], color='firebrick', linewidth=1.5)
ax.axhline(0.20, color='orange', linestyle='--', linewidth=1, label='20% crisis threshold')
ax.set_title('Mean % Population in IPC Phase 3+ (Crisis or Worse) - South Sudan', fontsize=13)
ax.set_xlabel('Date'); ax.set_ylabel('Mean % in Phase 3+'); ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/04_ipc_phase3plus_trend.png', dpi=150)
plt.show()

**[TODO - interpret.]** Is food insecurity improving or worsening? When does it breach the 20% threshold? Does this correlate with the conflict peaks you saw above?

In [ ]:
# 2e. Top 10 counties by food insecurity
top = ipc_cur_eda[ipc_cur_eda['Phase']=='3+'].groupby('Area')['Percentage'].mean().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(9,4))
top.plot(kind='barh', ax=ax, color='firebrick')
ax.set_title('Top 10 Counties by Mean IPC Phase 3+ Percentage', fontsize=13)
ax.set_xlabel('Mean % population in Phase 3+')
plt.tight_layout()
plt.savefig('../reports/figures/05_top_counties_food_insecurity.png', dpi=150)
plt.show()

**[TODO - interpret.]** Are the most food-insecure counties also among the most conflict-affected? This matters for your discussion section.

## 3. Preprocessing & Feature Engineering

All cleaning decisions are implemented in `src/preprocessing.py` with full documentation of the reasoning. This section applies those functions and verifies results at each step.

### 3a. County name standardisation

In [ ]:
acled = standardize_county_names(acled_raw.copy(), 'admin2')
ipc   = standardize_county_names(ipc_raw.copy(), 'Area')
ipc   = filter_ipc(ipc)

gaps = find_unmatched_counties(acled, ipc)
print('Only in ACLED (after fixes):', gaps['only_in_acled'])
print('Only in FEWS NET (after fixes):', gaps['only_in_ipc'])
print('IPC rows after filtering:', len(ipc))

**Verification:** Three fixes applied (Kajo Keji->Kajo-Keji, Raja->Raga, Yei->Yei County). Six intentional gaps remain — each is explained in `data/README.md` and must be discussed in the report's Methodology section.

**[TODO - paste the printed output here and confirm each remaining gap matches its documented reason.]**

### 3b. Aggregation and pivot

In [ ]:
acled_monthly = aggregate_acled_to_county_month(acled)
ipc_wide      = pivot_ipc_wide(ipc)

print('ACLED monthly shape:', acled_monthly.shape)
print('IPC wide shape:     ', ipc_wide.shape)
acled_monthly.head(3)

In [ ]:
ipc_wide.head(3)

### 3c. Merge with 1-month conflict lag

In [ ]:
# lag_months=1: conflict features from month T predict IPC assessment in month T+1.
# Avoids leakage. Reflects realistic early-warning scenario.
# Experiment 8 will test lag_months=2.
merged = merge_conflict_food_security(acled_monthly, ipc_wide, lag_months=1)
print('Merged shape:', merged.shape)
print('Rows with conflict match (event_count > 0):', (merged['event_count']>0).sum(), '/', len(merged))
print('Missing values:')
print(merged.isnull().sum()[merged.isnull().sum()>0])
merged.head()

### 3d. Target variable definition

In [ ]:
# Binary target: crisis_plus = 1 if >= 20% of county population is in IPC Phase 3+.
# The 20% threshold is the IPC's own operational trigger for emergency response
# (IPC Global Partners, 2021, Technical Manual v3.1).
merged['crisis_plus'] = (merged['pct_phase3plus'] >= 0.20).astype(int)
print('Target distribution:')
print(merged['crisis_plus'].value_counts())
print('Class balance (fraction crisis):', round(merged['crisis_plus'].mean(), 3))

**[TODO - interpret the class balance.]** Is the dataset imbalanced? This directly motivates class_weight='balanced' in Experiment 3 and the use of F1/AUC rather than raw accuracy as primary metrics.

### 3e. Feature matrix and chronological train/test split

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

conflict_features   = ['event_count','fatalities_sum','fatalities_max',
                        'battles_count','violence_civilians_count','protests_count']
geographic_features = ['Level 1']
feature_cols        = conflict_features + geographic_features

# Chronological split - NOT random.
# Train on earlier years, test on most recent years.
# A random split would let future months leak into training.
merged['year_month_dt'] = merged['year_month'].dt.to_timestamp()
train_mask = merged['year_month_dt'] <  '2023-01-01'
test_mask  = merged['year_month_dt'] >= '2023-01-01'

X_train = merged.loc[train_mask, feature_cols]
y_train = merged.loc[train_mask, 'crisis_plus']
X_test  = merged.loc[test_mask,  feature_cols]
y_test  = merged.loc[test_mask,  'crisis_plus']

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train: {merged.loc[train_mask,"year_month_dt"].min().date()} -> {merged.loc[train_mask,"year_month_dt"].max().date()}')
print(f'Test:  {merged.loc[test_mask, "year_month_dt"].min().date()} -> {merged.loc[test_mask, "year_month_dt"].max().date()}')
print(f'Train class balance: {y_train.mean():.3f}  |  Test class balance: {y_test.mean():.3f}')

In [ ]:
numeric_pipe    = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
categorical_pipe = Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor = ColumnTransformer([
    ('num', numeric_pipe,    conflict_features),
    ('cat', categorical_pipe, geographic_features),
])

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)
print('Processed features - Train:', X_train_proc.shape, '  Test:', X_test_proc.shape)

## 4. Classical Machine Learning Experiments

Four experiments using scikit-learn, progressing from a simple baseline to tuned ensemble methods. Log every run in `docs/experiment_log.md` as you go.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, ConfusionMatrixDisplay)

def evaluate_clf(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:,1] if hasattr(model,'predict_proba') else None
    print(f'\n{"="*55}\n  {name}\n{"="*55}')
    print(classification_report(y_te, y_pred, target_names=['No Crisis','Crisis+']))
    if y_prob is not None:
        print(f'  AUC-ROC: {roc_auc_score(y_te, y_prob):.4f}')
    return model, y_pred, y_prob

### Experiment 1 — Logistic Regression (Baseline)

In [ ]:
lr, lr_pred, lr_prob = evaluate_clf(
    'Exp 1: Logistic Regression (C=1.0)',
    LogisticRegression(C=1.0, max_iter=1000, random_state=SEED),
    X_train_proc, y_train, X_test_proc, y_test
)
# TODO: log in docs/experiment_log.md
# TODO: interpret precision vs recall for Crisis+ class below

**[TODO - interpret Exp 1.]** What does precision vs recall for Crisis+ tell you? Is the model biased toward the majority class?

### Experiment 2 — Random Forest (Default Settings)

In [ ]:
rf_default, rf_pred, rf_prob = evaluate_clf(
    'Exp 2: Random Forest (n_estimators=100, default depth)',
    RandomForestClassifier(n_estimators=100, random_state=SEED),
    X_train_proc, y_train, X_test_proc, y_test
)
# TODO: compare to Exp 1. Does non-linearity improve recall for Crisis+?

**[TODO - interpret Exp 2 vs Exp 1.]**

### Experiment 3 — Random Forest (Tuned for Class Imbalance)

In [ ]:
rf_tuned, rf_tuned_pred, rf_tuned_prob = evaluate_clf(
    'Exp 3: Random Forest (max_depth=10, class_weight=balanced)',
    RandomForestClassifier(n_estimators=100, max_depth=10,
                           class_weight='balanced', random_state=SEED),
    X_train_proc, y_train, X_test_proc, y_test
)
# TODO: does class_weight fix recall? Does max_depth=10 reduce overfitting vs Exp 2?

**[TODO - interpret Exp 3 vs Exp 2.]**

### Experiment 4 — Gradient Boosting

In [ ]:
gb, gb_pred, gb_prob = evaluate_clf(
    'Exp 4: Gradient Boosting (n_estimators=200, lr=0.05)',
    GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                               max_depth=4, random_state=SEED),
    X_train_proc, y_train, X_test_proc, y_test
)
# TODO: is this the best classical result? Why might boosting outperform bagging here?

**[TODO - interpret Exp 4.]**

## 5. Deep Learning Experiments

Four experiments using TensorFlow/Keras with both Sequential and Functional APIs. A tf.data pipeline is used throughout (required by the assignment specification).

In [ ]:
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping

BATCH_SIZE = 32
EPOCHS     = 100

def make_dataset(X, y, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (X.astype('float32'), y.values.astype('float32'))
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=len(X), seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(X_train_proc, y_train, shuffle=True)
test_ds  = make_dataset(X_test_proc,  y_test,  shuffle=False)
n_features = X_train_proc.shape[1]

early_stop = EarlyStopping(monitor='val_loss', patience=10,
                           restore_best_weights=True, verbose=1)

print(f'Input features: {n_features}  |  Train batches: {len(train_ds)}')

### Experiment 5 — Sequential MLP (No Regularisation)

In [ ]:
model_seq1 = keras.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1,  activation='sigmoid'),
], name='Sequential_MLP_Baseline')

model_seq1.compile(optimizer='adam', loss='binary_crossentropy',
                   metrics=['accuracy', keras.metrics.AUC(name='auc')])
model_seq1.summary()

history5 = model_seq1.fit(
    train_ds, validation_data=test_ds,
    epochs=EPOCHS, callbacks=[early_stop], verbose=0
)
print('Best val AUC:', max(history5.history['val_auc']))

**[TODO - interpret Exp 5.]** Watch training vs val loss gap. If it widens, overfitting is occurring — that motivates Exp 6.

### Experiment 6 — Sequential MLP + Dropout + BatchNorm

In [ ]:
model_seq2 = keras.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid'),
], name='Sequential_MLP_Regularised')

model_seq2.compile(optimizer='adam', loss='binary_crossentropy',
                   metrics=['accuracy', keras.metrics.AUC(name='auc')])

history6 = model_seq2.fit(
    train_ds, validation_data=test_ds,
    epochs=EPOCHS, callbacks=[early_stop], verbose=0
)
print('Best val AUC:', max(history6.history['val_auc']))

**[TODO - interpret Exp 6 vs Exp 5.]** Did Dropout + BatchNorm close the train/val gap? At what cost to final AUC?

### Experiment 7 — Functional API (Multi-Branch Architecture)

In [ ]:
n_conflict_feats   = len(conflict_features)
n_geographic_feats = n_features - n_conflict_feats

conflict_in   = Input(shape=(n_conflict_feats,),   name='conflict_input')
geographic_in = Input(shape=(n_geographic_feats,), name='geographic_input')

x1 = layers.Dense(32, activation='relu')(conflict_in)
x1 = layers.Dropout(0.3)(x1)
x1 = layers.Dense(16, activation='relu')(x1)

x2 = layers.Dense(16, activation='relu')(geographic_in)
x2 = layers.Dropout(0.2)(x2)

merged_out = layers.Concatenate()([x1, x2])
out = layers.Dense(16, activation='relu')(merged_out)
out = layers.Dense(1,  activation='sigmoid')(out)

model_func = Model(inputs=[conflict_in, geographic_in], outputs=out,
                   name='Functional_MultiInput_MLP')
model_func.compile(optimizer='adam', loss='binary_crossentropy',
                   metrics=['accuracy', keras.metrics.AUC(name='auc')])
model_func.summary()

In [ ]:
# Build split tf.data datasets for the two-input Functional model
def make_split_dataset(X, y, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((
        {'conflict_input':   X[:, :n_conflict_feats].astype('float32'),
         'geographic_input': X[:, n_conflict_feats:].astype('float32')},
        y.values.astype('float32')
    ))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(X), seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_split_ds = make_split_dataset(X_train_proc, y_train, shuffle=True)
test_split_ds  = make_split_dataset(X_test_proc,  y_test,  shuffle=False)

history7 = model_func.fit(
    train_split_ds, validation_data=test_split_ds,
    epochs=EPOCHS, callbacks=[early_stop], verbose=0
)
print('Best val AUC:', max(history7.history['val_auc']))

**[TODO - interpret Exp 7.]** Does separating conflict and geographic branches improve performance? Why or why not?

### Experiment 8 — Best DL Architecture with lag=2 months

In [ ]:
# Re-run the best DL architecture with lag_months=2.
# Tests whether a 2-month lag better captures delayed conflict->food security effects.
merged_lag2 = merge_conflict_food_security(acled_monthly, ipc_wide, lag_months=2)
merged_lag2['crisis_plus'] = (merged_lag2['pct_phase3plus'] >= 0.20).astype(int)
merged_lag2['year_month_dt'] = merged_lag2['year_month'].dt.to_timestamp()

train2 = merged_lag2[merged_lag2['year_month_dt'] <  '2023-01-01']
test2  = merged_lag2[merged_lag2['year_month_dt'] >= '2023-01-01']

X_tr2 = preprocessor.transform(train2[feature_cols])
X_te2 = preprocessor.transform(test2[feature_cols])
y_tr2 = train2['crisis_plus']
y_te2 = test2['crisis_plus']

train_ds2 = make_dataset(X_tr2, y_tr2, shuffle=True)
test_ds2  = make_dataset(X_te2, y_te2, shuffle=False)

# TODO: replace model_seq2 with whichever of Exps 5-7 performed best
model_lag2 = keras.models.clone_model(model_seq2)
model_lag2.compile(optimizer='adam', loss='binary_crossentropy',
                   metrics=['accuracy', keras.metrics.AUC(name='auc')])

history8 = model_lag2.fit(
    train_ds2, validation_data=test_ds2,
    epochs=EPOCHS, callbacks=[early_stop], verbose=0
)
print('Best val AUC (lag=2):', max(history8.history['val_auc']))

**[TODO - interpret Exp 8.]** Does lag=2 outperform lag=1? What does this tell you about when conflict translates into food insecurity?

## 6. Model Evaluation & Error Analysis

For all models: learning curves, confusion matrices, and ROC curves. Interpret each one specifically in the markdown cells below them.

### 6a. Learning Curves (Deep Learning)

In [ ]:
def plot_lc(history, title, ax_loss, ax_auc):
    ax_loss.plot(history.history['loss'],     label='Train loss')
    ax_loss.plot(history.history['val_loss'], label='Val loss')
    ax_loss.set_title(title + ' - Loss'); ax_loss.legend()
    ax_auc.plot(history.history['auc'],     label='Train AUC')
    ax_auc.plot(history.history['val_auc'], label='Val AUC')
    ax_auc.set_title(title + ' - AUC'); ax_auc.legend()

fig, axes = plt.subplots(4, 2, figsize=(13,18))
plot_lc(history5, 'Exp 5 - Baseline MLP',      axes[0,0], axes[0,1])
plot_lc(history6, 'Exp 6 - Regularised MLP',   axes[1,0], axes[1,1])
plot_lc(history7, 'Exp 7 - Functional API',    axes[2,0], axes[2,1])
plot_lc(history8, 'Exp 8 - lag=2 months',      axes[3,0], axes[3,1])
plt.tight_layout()
plt.savefig('../reports/figures/06_learning_curves.png', dpi=150)
plt.show()

**[TODO - interpret each pair of curves.]** For each experiment: does training loss decrease steadily? Does validation loss diverge (overfitting)? How does Exp 6 (with Dropout + BN) compare to Exp 5?

### 6b. Confusion Matrices (All 8 Experiments)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18,8))

cml_results = {
    'Exp 1 LR':        (lr_pred,         y_test),
    'Exp 2 RF':        (rf_pred,          y_test),
    'Exp 3 RF Tuned':  (rf_tuned_pred,    y_test),
    'Exp 4 GradBoost': (gb_pred,          y_test),
}
for ax, (name, (pred, label)) in zip(axes[0], cml_results.items()):
    ConfusionMatrixDisplay(confusion_matrix(label, pred),
                           display_labels=['No Crisis','Crisis+']).plot(ax=ax, colorbar=False)
    ax.set_title(name, fontsize=9)

dl_results = [
    ('Exp 5 Baseline MLP',  model_seq1, test_ds,       y_test),
    ('Exp 6 Regularised',   model_seq2, test_ds,       y_test),
    ('Exp 7 Functional',    model_func, test_split_ds, y_test),
    ('Exp 8 lag=2',         model_lag2, test_ds2,      y_te2),
]
for ax, (name, model, ds, label) in zip(axes[1], dl_results):
    prob = model.predict(ds, verbose=0).flatten()
    pred = (prob >= 0.5).astype(int)
    ConfusionMatrixDisplay(confusion_matrix(label, pred),
                           display_labels=['No Crisis','Crisis+']).plot(ax=ax, colorbar=False)
    ax.set_title(name, fontsize=9)

plt.suptitle('Confusion Matrices - All 8 Experiments', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/07_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

**[TODO - interpret.]** Which model has fewest false negatives (missed crises)? In a humanitarian early-warning context, false negatives are more costly than false alarms. How does class_weight='balanced' (Exp 3) change the false negative rate?

### 6c. ROC Curves (All 8 Experiments)

In [ ]:
fig, ax = plt.subplots(figsize=(9,7))

for name, prob, label in [
    ('Exp 1 LR',        lr_prob,        y_test),
    ('Exp 2 RF',        rf_prob,        y_test),
    ('Exp 3 RF Tuned',  rf_tuned_prob,  y_test),
    ('Exp 4 GradBoost', gb_prob,        y_test),
]:
    fpr, tpr, _ = roc_curve(label, prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(label,prob):.3f})')

for name, model, ds, label in dl_results:
    prob = model.predict(ds, verbose=0).flatten()
    fpr, tpr, _ = roc_curve(label, prob)
    ax.plot(fpr, tpr, linestyle='--',
            label=f'{name} (AUC={roc_auc_score(label,prob):.3f})')

ax.plot([0,1],[0,1],'k:', label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves - All 8 Experiments')
ax.legend(fontsize=7, loc='lower right')
plt.tight_layout()
plt.savefig('../reports/figures/08_roc_curves.png', dpi=150)
plt.show()

**[TODO - interpret.]** Which model has the highest AUC? At what operating point on the ROC curve would you deploy this in a humanitarian context? Does the lag=2 model show a different curve shape from lag=1?

### 6d. Bias-Variance Analysis & Dataset Limitations

**[TODO - write in your own words. Address:]**

**Bias-variance:** Which models underfit (high train + val error)? Which overfit (low train, high val error)? How do ensemble methods and regularised DL compare to Logistic Regression on this spectrum?

**Dataset limitations to discuss honestly:**
1. ACLED date gap - ends June 2025, not 2026. Most recent conflict trends are absent.
2. Six county-name gaps mean some counties have food security labels but zero conflict features, inflating the zero-conflict category.
3. ACLED underreporting - events in remote, low-connectivity areas are systematically underrepresented.
4. IPC assessment frequency - IPC assessments are quarterly, not monthly; many months are interpolated.
5. Confounding variables - drought, flooding, and seasonal factors affect both conflict and food security but are absent from the model.

**Proposed improvements:** Adding CHIRPS rainfall data, using population_best to weight events, testing LSTM with a longer lookback window, incorporating cross-border conflict from Sudan.

## 7. Conclusions

**[TODO - write in your own words.]**

Summarise: which model performed best and why? Does conflict data provide a meaningful early-warning signal for food insecurity in South Sudan? What are the practical implications for humanitarian actors and peacebuilding organisations?

Connect back to your mission: a technically sound model that cannot be used by a humanitarian coordinator in practice is not aligned with the goal of fostering enduring peace and sustainable development in South Sudan.